# SQLite Database Verification - Battery Cells
## FastAPI Backend Setup with SQLAlchemy ORM

This notebook verifies the successful import of 384 battery cell records into SQLite database.

In [1]:
# Import required libraries
import sys
import pandas as pd
from pathlib import Path

# Add backend to path so we can import app modules
sys.path.insert(0, str(Path.cwd()))

from app.db.database import SessionLocal
from app.models.cellule import Cellule
import statistics

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


## Database Summary Statistics

In [2]:
# Connect to database and get statistics
db = SessionLocal()

# Total count
total = db.query(Cellule).count()
print(f"✅ Total cells in database: {total}")

# Get types distribution
types_dist = {}
for cell in db.query(Cellule).all():
    types_dist[cell.type_cellule] = types_dist.get(cell.type_cellule, 0) + 1

print("\n📊 Distribution by Cell Type:")
for cell_type, count in sorted(types_dist.items()):
    pct = (count / total) * 100
    print(f"  • {cell_type:15s}: {count:4d} cells ({pct:5.1f}%)")

✅ Total cells in database: 384

📊 Distribution by Cell Type:
  • Cylindrical    :   99 cells ( 25.8%)
  • Pouch          :  151 cells ( 39.3%)
  • Prismatic      :  134 cells ( 34.9%)


## Display All Imported Data

In [3]:
# Query all cells and convert to DataFrame
all_cells = db.query(Cellule).all()

# Create DataFrame
data = []
for cell in all_cells:
    data.append({
        'id': cell.id,
        'nom': cell.nom,
        'longueur_mm': cell.longueur_mm,
        'largeur_mm': cell.largeur_mm,
        'hauteur_mm': cell.hauteur_mm,
        'masse_g': cell.masse_g,
        'tension_nominale': cell.tension_nominale,
        'capacite_ah': cell.capacite_ah,
        'courant_max_a': cell.courant_max_a,
        'type_cellule': cell.type_cellule,
        'taux_swelling_pct': cell.taux_swelling_pct
    })

df_database = pd.DataFrame(data)
print(f"DataFrame shape: {df_database.shape}")
print(f"\nFirst 20 rows of imported data:")
df_database.head(20)

DataFrame shape: (384, 11)

First 20 rows of imported data:


,id,nom,longueur_mm,largeur_mm,hauteur_mm,masse_g,tension_nominale,capacite_ah,courant_max_a,type_cellule,taux_swelling_pct
0,1,LP925572,72.00,55.00,9.20,72.0,3.70,4.00,8.0,Pouch,0.08
1,2,MP174565,70.00,45.50,18.10,103.0,3.75,4.80,10.0,Prismatic,0.03
2,3,MP174565xtd,68.50,45.30,18.65,97.0,3.65,4.00,8.0,Prismatic,0.03
3,4,SLPB11543140H5,142.50,43.50,11.70,128.0,3.70,5.00,150.0,Pouch,0.08
4,5,CE175-360 Moxie+,253.00,172.00,5.80,430.0,3.60,17.50,17.5,Pouch,0.08
5,6,LTO66160H/40Ah,66.00,66.00,160.00,1250.0,2.30,40.00,240.0,Cylindrical,0.00
6,7,HJLFP48173170E-120Ah,165.00,174.00,48.00,2860.0,3.20,120.00,240.0,Prismatic,0.03
7,8,9299130N,130.00,98.00,9.05,260.0,3.70,13.00,39.0,Pouch,0.08
8,9,SDI 94Ah,173.00,125.00,45.00,2100.0,3.68,94.00,150.0,Prismatic,0.03
9,10,ICP225054S,56.00,51.00,23.50,155.0,3.70,5.00,1.0,Prismatic,0.03


## Data Integrity Verification

In [4]:
# Check for NULL values and data integrity
print("✓ Data Integrity Checks:\n")

# Check for NULL values
null_checks = {
    'nom': df_database['nom'].isnull().sum(),
    'longueur_mm': df_database['longueur_mm'].isnull().sum(),
    'largeur_mm': df_database['largeur_mm'].isnull().sum(),
    'hauteur_mm': df_database['hauteur_mm'].isnull().sum(),
    'masse_g': df_database['masse_g'].isnull().sum(),
    'tension_nominale': df_database['tension_nominale'].isnull().sum(),
    'capacite_ah': df_database['capacite_ah'].isnull().sum(),
    'courant_max_a': df_database['courant_max_a'].isnull().sum(),
    'type_cellule': df_database['type_cellule'].isnull().sum(),
    'taux_swelling_pct': df_database['taux_swelling_pct'].isnull().sum(),
}

print("NULL Value Checks:")
for column, count in null_checks.items():
    status = "✓ PASS" if count == 0 else f"❌ FAIL ({count} NULLs)"
    print(f"  {column:20s}: {status}")

# Check data types
print(f"\nData Types:")
print(df_database.dtypes)

# Statistical summary
print(f"\n✓ Statistical Summary:")
print(df_database.describe())

✓ Data Integrity Checks:

NULL Value Checks:
  nom                 : ✓ PASS
  longueur_mm         : ✓ PASS
  largeur_mm          : ✓ PASS
  hauteur_mm          : ✓ PASS
  masse_g             : ✓ PASS
  tension_nominale    : ✓ PASS
  capacite_ah         : ✓ PASS
  courant_max_a       : ✓ PASS
  type_cellule        : ✓ PASS
  taux_swelling_pct   : ✓ PASS

Data Types:
id                     int64
nom                      str
longueur_mm          float64
largeur_mm           float64
hauteur_mm           float64
masse_g              float64
tension_nominale     float64
capacite_ah          float64
courant_max_a        float64
type_cellule             str
taux_swelling_pct    float64
dtype: object

✓ Statistical Summary:
               id  longueur_mm  largeur_mm  hauteur_mm       masse_g  \
count  384.000000   384.000000  384.000000  384.000000    384.000000   
mean   192.500000   160.560469  108.545885   48.089609   1711.262760   
std    110.995495   124.802598   83.509882   56.016489   44

In [5]:
# Close database session
db.close()

print("\n✅ Database Verification Complete!")
print("="*80)
print("SUCCESS: All 384 battery cells successfully imported into SQLite database")
print("="*80)


✅ Database Verification Complete!
SUCCESS: All 384 battery cells successfully imported into SQLite database
